# Suspicious-charge intake · initial baseline evaluation

Roberto · 26 September 2026 · Contract and runnable evaluation v0.1

This notebook compares **handoff-only** and **deterministic checklist** references on 42 authored Spanish/Portuguese decision-point cases. These fixtures are not sampled customer traffic, independently blinded gold labels, or evidence of reduced fraud. See `Docs/intake/customer-and-measurement-contract.md` for the customer boundary, complete-intake definition and KPI dictionary.

**Intended customer:** an authenticated existing customer reporting an unrecognized transaction through the web experience. Evidence collection and human review are the goal; no autonomous fraud determination, refund or card blocking is implemented.

**Read the rates correctly:** the product primary KPI is safe complete intake per in-scope conversation episode. This initial harness measures the next decision and completion-ready turns. It does not estimate episode completion, abandonment, operating cost or real service latency.

In [ ]:
from pathlib import Path
import sys, json, hashlib, platform
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / 'evals/intake').is_dir(): ROOT = ROOT.parent
assert (ROOT / 'evals/intake').is_dir(), 'Launch from the repository or notebooks directory'
sys.path.insert(0,str(ROOT))
from evals.intake.run import evaluate
corpus_bytes=(ROOT/'evals/intake/cases.json').read_bytes()
corpus=json.loads(corpus_bytes)
results=evaluate(corpus)
results['provenance']={'executed_utc':datetime.now(timezone.utc).isoformat(), 'python':platform.python_version(),
    'corpus_sha256':hashlib.sha256(corpus_bytes).hexdigest(),
    'code_sha256':hashlib.sha256((ROOT/'evals/intake/baseline.py').read_bytes()+(ROOT/'evals/intake/run.py').read_bytes()).hexdigest()}
OUT=ROOT/'data_foundation/runs/intake-evaluation';OUT.mkdir(parents=True,exist_ok=True)
(OUT/'notebook-results.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')
summary=pd.DataFrame(results['summary']); cases=pd.DataFrame(results['cases'])
def show(frame):
    display(frame.style.format(precision=3,na_rep='Not measured').hide(axis='index'))
print(json.dumps(results['provenance'],indent=2))


## Method and denominator checks

Development contains 18 cases across nine scenario families. Evaluation contains 24 cases across 12 different families, balanced by language. Translations within a family remain in the same split and are correlated; 42 cases do not represent 42 independent behavior families. The corpus was authored with knowledge of the rules and frozen before its first scored run. Safety-review fixes were applied after that run, so the reused evaluation split is a regression suite, not an untouched holdout. Team review and a separately authored unseen test set are still needed.

Both references receive the same messages, fixture transactions and trusted session context. Expected actions are never passed into the baseline. The scorer compares returned evidence to the fixture source. A completion without validated confirmation, wrong ownership or a prohibited requested action fails the safety check.

In [ ]:
inventory=pd.DataFrame(corpus['cases'])
show(inventory.groupby(['split','language']).agg(cases=('case_id','count'),families=('family','nunique')).reset_index())
assert not set(inventory.query("split == 'development'").family) & set(inventory.query("split == 'evaluation'").family)
assert summary.correct.le(summary.cases).all()
assert summary.safe_complete.le(summary.completion_ready).all()
show(summary[['split','baseline','language','cases','correct','unsafe']])


## Results: initial references

A correct clarification is a successful next action, not a completed intake. The completion-ready denominator is intentionally separate. Zero observed unsafe outcomes on this small corpus does not demonstrate production safety. The handoff-only reference still applies identity and narrow phrase checks; it does not blindly escalate unrelated requests.

In [ ]:
evaluation=summary.query("split == 'evaluation' and language == 'all'")
show(evaluation[['baseline','correct','cases','correct_rate','safe_complete','completion_ready','completion_ready_rate']])
show(summary.query("split == 'evaluation'")[['baseline','language','unsafe','cases','missed_handoff','required_handoff','unnecessary_handoff','not_required_handoff']])
fig,ax=plt.subplots(figsize=(9,4))
ax.barh(evaluation.baseline,evaluation.correct_rate,color=['#667085','#2463A5'])
for i,row in enumerate(evaluation.itertuples()): ax.text(row.correct_rate+.01,i,f'{row.correct}/{row.cases}',va='center')
ax.set(xlim=(0,1.12),xlabel='Correct next action / all evaluation cases',title='Initial authored evaluation · 24 decision points · 26 September 2026')
ax.xaxis.set_major_formatter(PercentFormatter(1));ax.spines[['top','right']].set_visible(False)
fig.tight_layout();plt.show();plt.close(fig)


## Failure analysis and the next iteration

Inspect case-level failures before adding rules. Evaluation failures remain visible; do not add these exact phrases and report the same set as unseen evidence. Relative dates, implied currencies and grouping separators intentionally trigger clarification. The next learned extractor should be tested with independently reviewed ES/PT cases covering paraphrases, negation, ambiguity, mixed-language input, unauthorized requests and tool failures.

In [ ]:
failures=cases.query("split == 'evaluation' and baseline == 'checklist' and not correct")
show(pd.DataFrame([{'case_id':r.case_id,'language':r.language,'expected':r.gold['action'],'actual':r.prediction['action'],'safe':r.safe} for r in failures.itertuples()]))
check=evaluation.loc[evaluation.baseline.eq('checklist')].iloc[0]
ref=evaluation.loc[evaluation.baseline.eq('handoff')].iloc[0]
display(Markdown(f"**Measured:** checklist {int(check.correct)}/{int(check.cases)} correct decisions; handoff reference {int(ref.correct)}/{int(ref.cases)}. Checklist safely completed {int(check.safe_complete)}/{int(check.completion_ready)} completion-ready turns. These are component results on authored fixtures, not the product episode KPI."))


## Latency, cost and source integration

The timings below are one local call per fixture, without a network, model or real service. They are a reproducibility diagnostic only. Operating cost and episode completion remain unmeasured. The optional `python -m evals.intake.source_smoke` command separately checks owned-transaction confirmation and rejection of a foreign confirmation against the read-only full-data exploration cache; that smoke test is not a benchmark or a claim of representative coverage.

In [ ]:
show(evaluation[['baseline','latency_p50_ms','latency_p95_ms','operating_cost','episode_completion_rate']])
smoke=OUT/'source-smoke.json'
if smoke.exists(): display(Markdown('**Separate source integration check:** '+json.loads(smoke.read_text())['scope']))
else: print('Optional source integration smoke not run in this workspace.')


## Recommended handoff to the team

1. Confirm the proposed customer and intent boundaries; recognized billing disputes are separate from unrecognized charges.
2. Review every gold scenario with Andrés/Lucas, then author a fresh unseen ES/PT set and multi-turn episodes.
3. Compare a learned extractor with these fixed references on that same workload; keep deterministic identity, evidence and action checks outside the model.
4. Measure complete episodes, required/incorrect handoffs and actual tool/model latency and cost once the service exists. Set improvement targets from reviewed evidence rather than inventing them.

**Sources:** current team direction supplied by Roberto; Lucas's architecture decision and Notion 26/09 update; prior reviewed exploration in `reports/2026-09-26/`; authored cases and executable rules in `evals/intake/`. No historical complaint-to-transaction outcomes or financial losses are inferred.